In [ ]:
# =========================================================================
# LTX-Video (Lightricks 2B DiT) 高保真零畸變圖生影 Worker (Version 19)
# =========================================================================
import os, sys, io, time, gc, json, urllib.request
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, psutil
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
from PIL import Image

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

print("📦 正在安裝與確認 LTX-Video 視頻生成專用依賴庫...")
import subprocess
subprocess.run(["pip", "install", "-q", "diffusers>=0.41.0", "transformers>=5.17.0", "accelerate", "imageio[ffmpeg]"], check=False)

from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

model_id = "Lightricks/LTX-Video"
print(f"🎬 [新世代 2B DiT 頂級寫實視頻模型] 正在載入: {model_id}...")

# 啟用 bfloat16 精度載入，並啟用逐組件 CPU Offload (顯存僅需 ~6GB，徹底杜絕 OOM)
pipe = LTXImageToVideoPipeline.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
)
print("⚙️ 啟用 Model CPU Offload 節能模式 (零 OOM 保證)...")
pipe.enable_model_cpu_offload()

# 載入通過 7 項品質檢驗的首幀 2K 母圖
mother_url = "https://raw.githubusercontent.com/Chenponpon/colab-ai-media-lab/main/assets/mother_images/portrait_2.jpg"
print(f"📥 正在從 GitHub 載入母圖: {mother_url} ...")
urllib.request.urlretrieve(mother_url, "input_mother.jpg")
input_img = Image.open("input_mother.jpg").convert("RGB")
orig_w, orig_h = input_img.size
print(f"   首幀原始解析度: {orig_w}x{orig_h}")

# 鎖定 LTX-Video 直式超清規格 (480x640，寬高均能被 32 整除，長寬比 0.75 契合母圖，絕不拉伸)
vid_w, vid_h = 480, 640
input_img = input_img.resize((vid_w, vid_h), Image.LANCZOS)
print(f"   縮放至視頻標準解析度: {vid_w}x{vid_h}")

# 精準微動態 Prompt：專注於呼吸律動、香肩微動與自然微笑，嚴格禁止手部抬起摸臉摸頭
motion_prompt = (
    "Three young breathtakingly gorgeous French-Vietnamese Eurasian women sitting elegantly on crisp white bed, "
    "gentle natural chest breathing movement, subtle shoulder movement, warm alluring smiles, "
    "soft natural eye blinks looking at the camera, photorealistic live-action motion, "
    "8k quality, cinematic morning bedroom lighting, perfectly stable angelic faces, hands resting naturally flat on thighs."
)
negative_prompt = (
    "hands touching face, hands touching hair, raising arms, morphing face, distorted eyes, "
    "deformed fingers, melting hands, blurry face, jerky motion, bad anatomy, low quality, unnatural warping"
)

# 幀數滿足 8k + 1 規則：k=8 => 65 幀 (65 / 16fps = 4.06 秒穩定微動態)
total_frames = 65
target_fps = 16
num_steps = 40
print(f"⏳ 正在啟動 LTX-Video 高保真圖生影管線 ({total_frames} 幀 / {num_steps} 步 / {target_fps}fps 電影級微動態)...")
t0 = time.time()
generator = torch.manual_seed(42)
video_frames = pipe(
    image=input_img,
    prompt=motion_prompt,
    negative_prompt=negative_prompt,
    width=vid_w,
    height=vid_h,
    num_frames=total_frames,
    num_inference_steps=num_steps,
    guidance_scale=3.0,
    generator=generator
).frames[0]

dt = round(time.time() - t0, 2)
print(f"✨ 視頻生成完畢！耗時: {dt} 秒，正在輸出 MP4...")

out_vid_name = "changing_clothes_video.mp4"
export_to_video(video_frames, out_vid_name, fps=target_fps)
vid_sz_mb = round(os.path.getsize(out_vid_name) / (1024 * 1024), 2)
print(f"🎉 影片已成功輸出為 {out_vid_name} (體積: {vid_sz_mb} MB)！")

bench = {
    "engine": "Lightricks/LTX-Video (2B DiT Image-to-Video)",
    "gpu": gpu_name,
    "elapsed_sec": dt,
    "frames": total_frames,
    "fps": target_fps,
    "steps": num_steps,
    "resolution": f"{vid_w}x{vid_h}",
    "video_file": out_vid_name,
    "file_size_mb": vid_sz_mb
}
with open("video_benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench, bf, indent=2)
